## Instalação das bibliotecas

- OpenAI: comunicação com o modelo de IA utilizado na análise das earnings calls.
- Pydantic: validação e estruturação das respostas em formato JSON.
- Python-dotenv: gerenciamento seguro de chaves e variáveis de ambiente.
- Pandas: manipulação e organização de dados para análise e geração de relatórios.

Essas bibliotecas serão utilizadas para processar transcrições, extrair insights estruturados e gerar os outputs solicitados no case.

In [1]:
!pip install openai pydantic python-dotenv pandas

## Organização do projeto

 Criando as pastas que serão utilizadas durante o desenvolvimento do Case 1.

* A pasta "data" armazenará as transcrições das earnings calls.

* A pasta "outputs" armazenará os arquivos gerados pela análise, como o JSON estruturado e o relatório em Markdown.



In [4]:
import os

os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)



## Upload da transcrição

Nesta etapa vou enviar a transcrição para o ambiente do Google Colab.

O objetivo é permitir que o Python leia o conteúdo do documento e o utilize posteriormente para gerar as análises solicitadas pelo case.

In [5]:
from google.colab import files

uploaded = files.upload()

Saving case2.pdf to case2.pdf


## Leitura do PDF

Nesta etapa vou instalar uma biblioteca capaz de ler o conteúdo textual do PDF.

O objetivo é extrair a transcrição da earnings call para que possamos analisá-la posteriormente com Python e IA.

In [8]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.3/346.3 kB 26.8 MB/s eta 0:00:00


## Extração do texto da transcrição

Nesta etapa vou abrir o PDF e extrair todo o texto da earnings call.

O texto extraído será armazenado em uma variável para ser utilizado posteriormente na geração das análises solicitadas pelo case.

In [11]:
from pypdf import PdfReader

arquivo = "case2.pdf"

reader = PdfReader(arquivo)

texto = ""

for pagina in reader.pages:
    texto += pagina.extract_text() + "\n"

print("Total de caracteres:", len(texto))
print("\nPrimeiros 1000 caracteres:\n")
print(texto[:1000])

Total de caracteres: 58780

Primeiros 1000 caracteres:

 
1 
 
PÚBLICA 
 
   
 
Webcast – 1st Quarter Results 2026 
                  May 12th, 2026 
 
 
Eduardo De Nardi: 
 
Good morning, and welcome to Petrobras' webcast for investors and analysts about our 
results for the 1Q26. It's a pleasure to be here with you today.  
 
This event will be held in Portuguese and English. You can access the links to both 
languages on our Investor Relations website.  
 
All participants will be in listen-only mode during the Company’s presentation, and after 
that we will have a question and answer session.  
 
We have with us today Magda Chambriard, CEO of Petrobras; Angélica Laureano, 
Executive Logistics and Commercialization Director; Clarice Coppetti, Executive Director 
for Corporate Affairs; Fernando Melgarejo, Executive Director and Investor Relations, 
Flavio Bretanha Freire representing the Engineering Technology and Innovation 
Director; Rica rdo Wagner, Governance and Compliance Execu

## Salvando a transcrição em TXT

Nesta etapa vou salvar o texto extraído do PDF em um arquivo .txt.

Isso facilita o processamento posterior e permite reutilizar a transcrição sem precisar ler o PDF novamente.

In [12]:
with open("transcricao.txt", "w", encoding="utf-8") as arquivo:
    arquivo.write(texto)

print("Transcrição salva com sucesso!")

Transcrição salva com sucesso!


## Estrutura da saída

Antes de solicitar a análise para a IA, vou definir o formato da resposta esperada.





In [13]:
estrutura_saida = {
    "management_tone": "",
    "tone_evidence": [],

    "guidance_changes": [],

    "critical_questions": [],

    "red_flags": [],

    "surprise_score": "",

    "executive_summary": ""
}

print(estrutura_saida)

{'management_tone': '', 'tone_evidence': [], 'guidance_changes': [], 'critical_questions': [], 'red_flags': [], 'surprise_score': '', 'executive_summary': ''}


## Modelagem da resposta com Pydantic

Nesta etapavb vou ultilizar Pydantic para definir a estrutura esperada da análise.

O objetivo é garantir que a saída da aplicação siga um formato padronizado e validado, facilitando a manutenção do código e a integração com futuras etapas do pipeline.

In [14]:
from pydantic import BaseModel
from typing import List


class EarningsCallAnalysis(BaseModel):
    management_tone: str
    tone_evidence: List[str]

    guidance_changes: List[str]

    critical_questions: List[str]

    red_flags: List[str]

    surprise_score: str

    executive_summary: str


print("Modelo criado com sucesso!")

Modelo criado com sucesso!


## Configuração da chave da OpenAI

Nesta etapa vou armazenar a chave da OpenAI em uma variável.

Ela será utilizada para autenticar as requisições ao modelo de IA responsável pela análise da earnings call.

In [15]:
OPENAI_API_KEY = **************************
print("Chave configurada com sucesso!")

Chave configurada com sucesso!


## Teste de conexão com a OpenAI

Nesta etapa vou validar a comunicação entre a aplicação e a API da OpenAI.

O objetivo é confirmar que a chave foi configurada corretamente e que o ambiente está apto a realizar chamadas para o modelo de IA.

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

resposta = client.responses.create(
    model="gpt-4.1-mini",
    input="Responda apenas: conexão realizada com sucesso"
)

print(resposta.output_text)

## Alteração do provedor de IA

Inicialmente, a solução foi desenvolvida utilizando a API da OpenAI. Durante os testes de integração, foi identificado um bloqueio relacionado à disponibilidade de créditos da conta utilizada para o desenvolvimento.

Como o objetivo deste case é demonstrar a arquitetura da solução, a engenharia de prompts, a estruturação das respostas e a capacidade analítica do sistema, optou-se pela utilização de um provedor alternativo de modelos de linguagem, mantendo a mesma lógica de processamento e os mesmos requisitos funcionais definidos para o projeto.

Essa alteração não impacta a arquitetura da aplicação nem os resultados esperados, alterando apenas o serviço responsável pela inferência do modelo.

In [17]:
# Migração para groq

!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 12.0 MB/s eta 0:00:00


In [46]:
import os
from groq import Groq

os.environ["GROQ_API_KEY"] = ""

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

print("Cliente Groq configurado com sucesso!")

Cliente Groq configurado com sucesso!


In [49]:
import os
from groq import Groq

client = Groq(
    api_key=""
)

## Teste de comunicação

Antes de processar a transcrição completa, realizamos um teste simples para validar a comunicação com a API da Groq.



In [50]:
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Responda apenas: conexão realizada com sucesso"
        }
    ],
    model="llama-3.3-70b-versatile"
)

print(chat_completion.choices[0].message.content)

Conexão realizada com sucesso.


## Análise do tom da administração

O primeiro objetivo é identificar o tom predominante da administração durante a earnings call.

A análise considera o conteúdo das respostas dos executivos, perspectivas apresentadas, planos de crescimento, gestão de riscos e expectativas futuras.

O resultado deve classificar o tom como Positivo, Neutro ou Negativo, acompanhado das principais evidências encontradas na transcrição.

In [20]:
prompt_tom = f"""
Você é um analista sell-side especializado em análise de earnings calls.

Analise a transcrição abaixo e responda:

1. Qual o tom predominante da administração?
   (Positivo, Neutro ou Negativo)

2. Liste 5 evidências textuais que justificam essa classificação.

3. Explique em até 10 linhas o motivo da classificação.

Transcrição:

{texto[:30000]}
"""

## Execução da análise

Nesta etapa o prompt será enviado ao modelo para identificação do tom predominante da administração e das evidências presentes na transcrição.

In [21]:
resposta = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt_tom
        }
    ],
    temperature=0.2
)

print(resposta.choices[0].message.content)

**1. Qual o tom predominante da administração?**

O tom predominante da administração é **Positivo**.

**2. Liste 5 evidências textuais que justificam essa classificação.**

1. "It's a pleasure to be here with you today" - Eduardo De Nardi (introdução ao webcast), indicando um tom acolhedor e positivo.
2. "I would like to congratulate all of our employees for the results that they delivered this quarter, which have made us so proud" - Magda Chambriard (CEO), expressando orgulho e satisfação com os resultados.
3. "Our oil and gas production has reached unprecedented levels" - Magda Chambriard (CEO), destacando o sucesso da empresa em termos de produção.
4. "We are very proud to say that we are having excellent results in terms of the sales of crude oil to international markets" - Magda Chambriard (CEO), enfatizando o sucesso da empresa em termos de vendas.
5. "We are now in the midst of a virtuous cycle" - Magda Chambriard (CEO), descrevendo a situação da empresa como positiva e de cres

## Registro do tom da administração

Nesta etapa armazenamos o resultado da análise em uma estrutura organizada.

O objetivo é iniciar a construção do JSON final do projeto, permitindo consolidar gradualmente todas as análises realizadas sobre a earnings call.

In [22]:
analise = {
    "management_tone": "Positivo",
    "tone_evidence": [
        "Our oil and gas production has reached unprecedented levels",
        "We are very proud to say that we are having excellent results in terms of the sales of crude oil",
        "We are now in the midst of a virtuous cycle",
        "I would like to congratulate all of our employees for the results that they delivered this quarter",
        "It's a pleasure to be here with you today"
    ]
}

print("Tom registrado com sucesso!")

Tom registrado com sucesso!


## Identificação de Red Flags

Nesta etapa buscamos identificar possíveis sinais de risco, incertezas ou desafios mencionados pela administração durante a earnings call.

Mesmo em calls com tom predominantemente positivo, é importante identificar fatores que possam impactar os resultados futuros, a geração de caixa, investimentos ou a execução da estratégia da companhia.

In [23]:
prompt_red_flags = f"""
Você é um analista de equity research.

Analise a transcrição abaixo e identifique:

1. As 5 principais Red Flags mencionadas ou sugeridas pela administração.
2. Explique por que cada uma delas representa um risco.
3. Classifique cada Red Flag como:
   - Baixa
   - Média
   - Alta

Retorne no formato:

Red Flag:
Nível:
Justificativa:

Transcrição:

{texto[:30000]}
"""

## Execução da análise de Red Flags

Nesta etapa o modelo avaliará a transcrição em busca de riscos operacionais, financeiros, estratégicos e de mercado que possam impactar a companhia.

In [24]:
resposta_redflags = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt_red_flags
        }
    ],
    temperature=0.2
)

print(resposta_redflags.choices[0].message.content)

Aqui estão as 5 principais Red Flags mencionadas ou sugeridas pela administração da Petrobras, juntamente com a justificativa e a classificação do nível de risco:

1. **Dependência do preço do petróleo**: A Petrobras menciona que o aumento do preço do petróleo pode afetar a empresa, mas também destaca que está trabalhando para mitigar esses efeitos.
Nível: Média
Justificativa: A dependência do preço do petróleo é um risco para a Petrobras, pois pode afetar a sua receita e lucratividade. No entanto, a empresa está trabalhando para diversificar sua produção e reduzir sua dependência do preço do petróleo.

2. **Risco de escassez de diesel e gasolina**: A Petrobras menciona que está trabalhando para aumentar a produção de diesel e gasolina, mas também destaca que há um risco de escassez desses produtos no mercado brasileiro.
Nível: Baixa
Justificativa: Embora haja um risco de escassez de diesel e gasolina, a Petrobras está trabalhando para aumentar a produção e reduzir a dependência de imp

## Registro das Red Flags

Nesta etapa consolidamos os principais riscos identificados durante a earnings call.

As Red Flags representam fatores que podem impactar a geração de caixa, a execução da estratégia, a rentabilidade futura ou a percepção dos investidores em relação à companhia.

In [25]:
analise["red_flags"] = [
    "Dependência de subsídios governamentais para parte da operação de combustíveis",
    "Alta volatilidade do preço do Brent e incerteza sobre cenário futuro",
    "Necessidade de importação de diesel no segundo semestre",
    "Projetos internacionais em México e Venezuela ainda em estágio inicial",
    "Baixa probabilidade de dividendos extraordinários no curto prazo"
]

print("Red Flags registradas com sucesso!")

Red Flags registradas com sucesso!


## Identificação das perguntas críticas dos analistas

Nesta etapa vou identificar os principais temas levantados pelos analistas durante a sessão de perguntas e respostas.

Essas perguntas são importantes porque revelam quais pontos geraram maior atenção ou preocupação por parte do mercado, permitindo compreender melhor os riscos, oportunidades e expectativas em relação à companhia.

In [26]:
prompt_perguntas = f"""
Você é um analista de Equity Research.

Analise a transcrição abaixo e identifique:

1. As 5 perguntas mais importantes feitas pelos analistas.
2. Quem fez a pergunta.
3. Qual era a principal preocupação ou interesse por trás da pergunta.
4. Resuma cada pergunta em no máximo 3 linhas.

Transcrição:

{texto[:30000]}
"""

## Execução da análise das perguntas



In [27]:
resposta_perguntas = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt_perguntas
        }
    ],
    temperature=0.2
)

print(resposta_perguntas.choices[0].message.content)

Aqui estão as 5 perguntas mais importantes feitas pelos analistas, juntamente com as informações solicitadas:

1. **Pergunta**: Qual é a perspectiva da Petrobras sobre o mercado downstream e como a empresa está lidando com as flutuações de preços?
**Quem fez a pergunta**: Leonardo Marcondes, Bank of America
**Preocupação principal**: Entender como a Petrobras está lidando com as flutuações de preços no mercado downstream e como isso afeta a empresa.
**Resumo**: A Petrobras está monitorando os preços e não está passando a volatilidade para os consumidores. A empresa está trabalhando para manter a sua participação de mercado no Brasil.

2. **Pergunta**: Como a Petrobras está lidando com a questão da auto-suficiência em diesel e gasoline no Brasil?
**Quem fez a pergunta**: Não especificado
**Preocupação principal**: Entender como a Petrobras está trabalhando para atingir a auto-suficiência em diesel e gasoline no Brasil.
**Resumo**: A Petrobras está trabalhando para atingir a auto-suficiê

A IA respondeu algumas perguntas de forma genérica e até repetiu temas. Como você está usando uma transcrição estruturada, vale a pena registrar perguntas mais específicas que realmente apareceram na call.

Pelas páginas da transcrição, eu destacaria:

Rodrigo Almeida (BTG) — Impacto dos subsídios governamentais e capital de giro.

Vicente Falanga (Bradesco) — Estratégia da Petrobras para a Braskem e expansão no México.

Gabriel Barra (Citi) — Possível aumento de CAPEX diante do cenário mais favorável para o petróleo.

Tasso Vasconcellos (UBS) — Como a Petrobras revisa seus planos diante da alta do Brent e da geração adicional de caixa.

Bruno Amorim (Goldman Sachs) — Possibilidade de dividendos extraordinários e prioridades para uso do caixa.

## Registro das perguntas críticas

Nesta etapa vou consilidar os principais questionamentos realizados pelos analistas durante a sessão de perguntas e respostas.

Essas perguntas refletem os temas de maior interesse do mercado e ajudam a compreender os fatores que podem impactar a percepção dos investidores sobre a companhia.

In [28]:
analise["critical_questions"] = [
    "Impacto dos subsídios governamentais sobre capital de giro e fluxo de caixa",
    "Estratégia para Braskem e oportunidades no México",
    "Possível aumento de CAPEX em função do cenário favorável do petróleo",
    "Critérios para revisão dos planos diante da alta do Brent",
    "Possibilidade de dividendos extraordinários e redução da dívida"
]

print("Perguntas críticas registradas com sucesso!")

Perguntas críticas registradas com sucesso!


## Surprise Score

Nesta etapa avalio os elementos da earnings call que podem ser considerados positivos ou negativos em relação às expectativas do mercado.

O objetivo é identificar fatos, decisões ou perspectivas que tenham potencial para alterar a percepção dos investidores sobre a companhia.

O resultado será utilizado para atribuir um Surprise Score qualitativo.

In [29]:
prompt_surprise = f"""
Você é um analista sell-side especializado em Petrobras.

Analise a transcrição abaixo e responda:

1. Quais foram os 5 pontos mais surpreendentes da call?
2. Cada ponto foi:
   - Positivo
   - Negativo
   - Neutro

3. Atribua um Surprise Score de 1 a 10.

4. Explique em até 10 linhas como chegou nessa nota.

Transcrição:

{texto[:30000]}
"""

## Execução do Surprise Score



In [30]:
resposta_surprise = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt_surprise
        }
    ],
    temperature=0.2
)

print(resposta_surprise.choices[0].message.content)

Aqui estão as respostas às suas perguntas:

1. Os 5 pontos mais surpreendentes da call:
 * A produção de petróleo da Petrobras alcançou 2,58 milhões de barris por dia, um recorde histórico.
 * A empresa está trabalhando para aumentar a produção de diesel e gasoline para atender à demanda brasileira.
 * A Petrobras está investindo em projetos de exploração e produção de petróleo e gás natural, incluindo o projeto Sergipe deepwater.
 * A empresa está trabalhando para reduzir a volatilidade dos preços de combustíveis no mercado brasileiro.
 * A Petrobras está comprometida em manter a disciplina de capital e a alocação eficiente de recursos.

2. Cada ponto foi:
 * Positivo: A produção de petróleo alcançou um recorde histórico, a empresa está trabalhando para aumentar a produção de diesel e gasoline, e está investindo em projetos de exploração e produção.
 * Neutro: A Petrobras está trabalhando para reduzir a volatilidade dos preços de combustíveis no mercado brasileiro.
 * Positivo: A empr

In [31]:
# Registro do Surprise Score


analise["surprise_score"] = "8/10"

analise["surprise_points"] = [
    "Produção recorde de 2,58 milhões de barris por dia",
    "Expansão da capacidade de diesel e gasolina",
    "Investimentos em novos projetos offshore",
    "Compromisso com disciplina de capital",
    "Resiliência operacional mesmo em cenário volátil"
]

print("Surprise Score registrado com sucesso!")

Surprise Score registrado com sucesso!



## Geração do Resumo Executivo

Nesta etapa será produzido um resumo executivo consolidando os principais pontos observados durante a earnings call.

O objetivo é fornecer uma visão rápida e objetiva sobre desempenho operacional, riscos, perspectivas futuras e percepção geral da administração.

In [32]:
prompt_resumo = f"""
Você é um analista sênior de Equity Research.

Com base na análise realizada, produza um resumo executivo de no máximo 15 linhas contendo:

- Visão geral da call
- Tom da administração
- Principais destaques positivos
- Principais riscos
- Questões levantadas pelos analistas
- Conclusão final

Tom:
{analise['management_tone']}

Red Flags:
{analise['red_flags']}

Perguntas:
{analise['critical_questions']}

Surprise Score:
{analise['surprise_score']}
"""

In [33]:
resumo = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt_resumo
        }
    ],
    temperature=0.2
)

print(resumo.choices[0].message.content)

**Resumo Executivo**

A call foi marcada por um tom positivo, com a administração demonstrando otimismo em relação ao desempenho futuro da empresa. Os principais destaques positivos incluem o crescimento da demanda por combustíveis e a expansão de projetos internacionais.

No entanto, foram destacados alguns riscos, incluindo a dependência de subsídios governamentais, a volatilidade do preço do Brent e a necessidade de importação de diesel. Os analistas questionaram a estratégia para a Braskem, oportunidades no México e a possibilidade de dividendos extraordinários.

A administração ressaltou a importância de monitorar o cenário de mercado e ajustar os planos conforme necessário. Com um Surprise Score de 8/10, a call foi considerada surpreendentemente positiva.

**Conclusão**: A empresa apresenta um cenário promissor, mas com riscos a serem monitorados. A administração demonstrou confiança no futuro da empresa, mas é fundamental acompanhar os desenvolvimentos futuros.


In [35]:

## Consolidação dos resultados
## Nesta etapa consolidamos todas as análises realizadas ao longo do projeto em uma única estrutura JSON.
## O objetivo é disponibilizar um resultado estruturado, reutilizável e compatível com futuras integrações, dashboards ou relatórios automatizados

analise["executive_summary"] = """
A call foi marcada por um tom positivo, com a administração demonstrando otimismo em relação ao desempenho futuro da empresa. Os principais destaques positivos incluem o crescimento da demanda por combustíveis e a expansão de projetos internacionais.

Foram identificados riscos relacionados à dependência de subsídios governamentais, volatilidade do Brent, necessidade de importação de diesel e incertezas em projetos internacionais.

Os analistas concentraram suas perguntas em dividendos, CAPEX, Braskem, expansão internacional e impactos do cenário macroeconômico.

A administração demonstrou confiança na execução da estratégia e manutenção da disciplina de capital.

Surprise Score: 8/10.
"""

print("Resumo executivo registrado com sucesso!")

Resumo executivo registrado com sucesso!


## Exportação do JSON

Esse arquivo representa a saída principal da aplicação e contém todos os insights gerados durante a análise da earnings call.

In [36]:
import json

with open("analise_petrobras.json", "w", encoding="utf-8") as arquivo:
    json.dump(
        analise,
        arquivo,
        ensure_ascii=False,
        indent=4
    )

print("JSON exportado com sucesso!")

JSON exportado com sucesso!



## Geração do Relatório Executivo

O objetivo é disponibilizar uma visão executiva dos resultados, permitindo uma leitura rápida por gestores, analistas e tomadores de decisão.

In [37]:
relatorio = f"""
# Petrobras Earnings Call Analysis

## Management Tone

{analise['management_tone']}

## Tone Evidence

{"".join(["- " + item + "\n" for item in analise['tone_evidence']])}

## Red Flags

{"".join(["- " + item + "\n" for item in analise['red_flags']])}

## Critical Questions

{"".join(["- " + item + "\n" for item in analise['critical_questions']])}

## Surprise Score

{analise['surprise_score']}

## Executive Summary

{analise['executive_summary']}
"""

with open("relatorio_executivo.md", "w", encoding="utf-8") as arquivo:
    arquivo.write(relatorio)

print("Relatório gerado com sucesso!")

Relatório gerado com sucesso!


## Download dos arquivos finais



In [38]:
from google.colab import files

files.download("analise_petrobras.json")
files.download("relatorio_executivo.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Evidências da análise

Para aumentar a confiabilidade dos resultados, serão armazenados trechos relevantes da transcrição que justificam as conclusões apresentadas.

Essa abordagem melhora a rastreabilidade dos insights e permite que o usuário valide os resultados diretamente na fonte original.

In [39]:
prompt_evidencias = f"""
Você é um analista de Equity Research.

Analise a transcrição abaixo e extraia:

1. 5 frases que demonstram tom positivo.
2. 3 frases que indiquem riscos ou incertezas.
3. 3 frases relacionadas a dividendos, CAPEX ou geração de caixa.

Retorne apenas os trechos exatos encontrados na transcrição.

Transcrição:

{texto[:30000]}
"""

In [40]:
evidencias = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt_evidencias
        }
    ],
    temperature=0
)

print(evidencias.choices[0].message.content)

Aqui estão as frases solicitadas:

**Frases com tom positivo:**

1. "It's a pleasure to be here with you today."
2. "We are happy once more to present this quarter's results, which have been marked by operational records."
3. "Our oil and gas production has reached unprecedented levels, which is the result of the exceptional effort we have made and a strong partnership between all the different areas of Petrobras."
4. "We are very proud to say that we are having excellent results in terms of the sales of crude oil to international markets."
5. "We are now in the midst of a virtuous cycle, where our investments are being converted into more production and higher added value products available on the market as well as higher revenue for the Company."

**Frases que indicam riscos ou incertezas:**

1. "However, this has already been surpassed in April, so it was a production of 2.58 million on average last year, which has already been surpassed as we reached a milestone of 2.73 million bar

In [41]:
relatorio_v2 = f"""
# Petrobras Earnings Call Analysis

## Management Tone

{analise['management_tone']}

## Evidências de Tom Positivo

- It's a pleasure to be here with you today.
- We are happy once more to present this quarter's results, which have been marked by operational records.
- Our oil and gas production has reached unprecedented levels.
- We are very proud to say that we are having excellent results in terms of the sales of crude oil.
- We are now in the midst of a virtuous cycle.

## Principais Red Flags

{"".join(["- " + item + "\n" for item in analise['red_flags']])}

## Evidências de Risco

- We might need to support these price increases with extended payment dates.
- We are always analyzing how prices are changing in the international market.
- We have to be sure that this market is still partly ours.

## Perguntas Críticas dos Analistas

{"".join(["- " + item + "\n" for item in analise['critical_questions']])}

## Evidências Financeiras

- We invested US$5 billion in the 1Q.
- Nearly 90% going to E&P projects.
- We closed the quarter with US$9.1 billion in cash.

## Surprise Score

{analise['surprise_score']}

## Executive Summary

{analise['executive_summary']}
"""

In [42]:
with open("relatorio_executivo_v2.md", "w", encoding="utf-8") as arquivo:
    arquivo.write(relatorio_v2)

print("Relatório V2 gerado com sucesso!")

Relatório V2 gerado com sucesso!


In [45]:
files.download("relatorio_executivo_v2.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>